**Notebook 04: H1 RFID and IoT23 Validation**


H1 RFID and IoT23 Validation

H1: Zero Trust implementation improves anomaly detection performance in RFID baggage tracking systems compared to traditional rule-based methods.

RQ1: How can Zero Trust improve anomaly detection in RFID baggage tracking systems?

This notebook runs more H1 checks using the simulated RFID scans and a separate public IoT23 network dataset. The RFID data is used to compare traditional rules with Zero Trust checks. The IoT23 data is used for a separate network traffic test.

BLUF: The notebook checks whether the added security rules find more abnormal activity. IoT23 contains network records, not real baggage scans, so the two results are kept separate.


Step 1: Connect and Review the Data

The prepared RFID file and IoT23 network data are opened from the project folder. The notebook checks the record counts, fields, and labels before starting the tests.


In [1]:


# Step 1: Connect and review the RFID and IoT23 data

# Load the libraries
from pathlib import Path
import pandas as pd

# Set the main project folder
main_folder = Path.cwd()

if not (main_folder / "06_Prepared_Data").exists():
    main_folder = main_folder.parent

# Set the data files
rfid_file = main_folder / "06_Prepared_Data/02_Prepared_RFID_Data.csv"
iot23_file = main_folder / "04_IoT_23/data/iot23_network_traffic.csv"

# Load both datasets
rfid_data = pd.read_csv(rfid_file)
iot23_data = pd.read_csv(iot23_file)

print("Both datasets are connected.")

# Review the RFID records
print("\nTotal RFID records:", len(rfid_data))
print("Total baggage journeys:", rfid_data["bag_id"].nunique())

print("\nRFID scan labels:")
print(rfid_data["final_label"].value_counts())

# Review the IoT23 records
print("\nTotal IoT23 records:", len(iot23_data))
print("Total IoT23 columns:", len(iot23_data.columns))

print("\nIoT23 fields:")
print(iot23_data.columns.tolist())

# Review the network labels
print("\nIoT23 network labels:")
print(iot23_data["label"].value_counts(dropna=False))

# Display sample network records
print("\nSample IoT23 records:")
print(iot23_data.head(3))

Both datasets are connected.

Total RFID records: 31575
Total baggage journeys: 5000

RFID scan labels:
final_label
Normal      30225
Abnormal     1350
Name: count, dtype: int64

Total IoT23 records: 23145
Total IoT23 columns: 23

IoT23 fields:
['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p', 'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes', 'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes', 'tunnel_parents', 'label', 'det_label']

IoT23 network labels:
label
Malicious    21222
Benign        1923
Name: count, dtype: int64

Sample IoT23 records:
             ts                 uid      id.orig_h  id.orig_p       id.resp_h  \
0  1.545404e+09   CrDn63WjJEmrWGjqf  192.168.1.195      41040  185.244.25.235   
1  1.545404e+09  CY9lJW3gh1Eje4usP6  192.168.1.195      41040  185.244.25.235   
2  1.545404e+09   CcFXLynukEDnUlvgl  192.168.1.195      41040  185.244.25.235   

   id.resp_p proto 

Step 2: Review the Security Conditions

The RFID records are checked for abnormal scans. The IoT23 records are checked for traffic labels, network types, and missing values. The known labels are kept for scoring the results, not for making the security decisions.


In [2]:


# Step 2: Review the RFID and IoT23 security conditions

# Review the RFID scan conditions
print("RFID scan conditions:")
print(rfid_data["scan_status"].value_counts(dropna=False))

# Review the detailed IoT23 security categories
print("\nIoT23 security categories:")
print(iot23_data["det_label"].value_counts(dropna=False).head(15))

# Compare network protocols by traffic label
print("\nNetwork protocols by traffic label:")
print(pd.crosstab(
    iot23_data["proto"],
    iot23_data["label"]
))

# Identify network measurements with missing values
network_fields = [
    "duration",
    "orig_bytes",
    "resp_bytes",
    "orig_pkts",
    "resp_pkts"
]

print("\nMissing network measurements:")
print(iot23_data[network_fields].isna().sum())

RFID scan conditions:
scan_status
Normal               30225
Spoofed Reader         375
Delayed                225
Duplicate              225
Out of Sequence        225
Invalid Location       225
Denial of Service       75
Name: count, dtype: int64

IoT23 security categories:
det_label
DDoS                         14394
C&C                           6706
-                             1923
PartOfAHorizontalPortScan      122
Name: count, dtype: int64

Network protocols by traffic label:
label  Benign  Malicious
proto                   
tcp       111      21220
udp      1812          2

Missing network measurements:
duration      0
orig_bytes    0
resp_bytes    0
orig_pkts     0
resp_pkts     0
dtype: int64


Step 3: Prepare the RFID Test Records

The baggage journeys are split to make a separate sample for another H1 check. Scans from the same bag stay in the same group. The original scan conditions and labels are not changed.


In [3]:
# Step 3: Prepare the RFID validation data

# Identify the unique baggage journeys
bag_ids = rfid_data["bag_id"].drop_duplicates()

# Select 25% of baggage journeys for validation
validation_bags = bag_ids.sample(
    frac=0.25,
    random_state=42
)

# Create the validation dataset
rfid_validation = rfid_data[
    rfid_data["bag_id"].isin(validation_bags)
].copy()

# Keep the remaining records separate
rfid_remaining = rfid_data[
    ~rfid_data["bag_id"].isin(validation_bags)
].copy()

# Review the validation sample
print("RFID validation records:", len(rfid_validation))

print(
    "Validation baggage journeys:",
    rfid_validation["bag_id"].nunique()
)

# Review normal and abnormal scans
print("\nValidation scan labels:")
print(rfid_validation["final_label"].value_counts())

# Review abnormal scan conditions
print("\nValidation scan conditions:")
print(rfid_validation["scan_status"].value_counts())

# Confirm that no baggage journey appears in both groups
overlap = set(rfid_validation["bag_id"]) & set(rfid_remaining["bag_id"])

print("\nOverlapping baggage journeys:", len(overlap))


RFID validation records: 8764
Validation baggage journeys: 1250

Validation scan labels:
final_label
Normal      8434
Abnormal     330
Name: count, dtype: int64

Validation scan conditions:
scan_status
Normal               8434
Spoofed Reader         96
Invalid Location       65
Out of Sequence        54
Duplicate              48
Delayed                46
Denial of Service      21
Name: count, dtype: int64

Overlapping baggage journeys: 0


Step 4: Apply the Two Security Checks

The traditional rules and Zero Trust checks from Notebook 03 are run on the same RFID scans. Traditional rules check required scan details, readers, locations, and order. Zero Trust adds more checks, including reader permission, timing, and repeated scans. The known answers are not used to decide which scans to flag.


In [4]:

# Step 4: Apply traditional and Zero Trust validation

# Create a working copy of the validation records
validation_data = rfid_validation.copy()

# Sort the records by bag and timestamp
validation_data = validation_data.sort_values(
    ["bag_id", "timestamp"]
).reset_index(drop=True)

# Set the recognized RFID readers
recognized_readers = [
    "RDR001", "RDR002", "RDR003", "RDR004",
    "RDR005", "RDR006", "RDR007"
]

# Set the required scan fields
required_fields = [
    "bag_id", "tag_id", "reader_id",
    "reader_location", "custody_point",
    "timestamp", "flight_id", "route_id", "scan_id"
]

# Apply the traditional security checks
validation_data["traditional_flagged"] = ~(
    validation_data[required_fields].notna().all(axis=1)
    & validation_data["reader_id"].isin(recognized_readers)
    & validation_data["location_match"].eq(1)
    & validation_data["sequence_match"].eq(1)
)

# Identify the preceding scan for each bag
previous_scan = validation_data.groupby("bag_id")[
    ["reader_id", "reader_location", "custody_point"]
].shift(1)

# Check for duplicate scans within one minute
duplicate_check = (
    validation_data["reader_id"].eq(previous_scan["reader_id"])
    & validation_data["reader_location"].eq(previous_scan["reader_location"])
    & validation_data["custody_point"].eq(previous_scan["custody_point"])
    & validation_data["scan_gap_minutes"].le(1)
)

# Apply the Zero Trust security checks
validation_data["zta_flagged"] = ~(
    validation_data[required_fields].notna().all(axis=1)
    & validation_data["reader_id"].eq(validation_data["expected_reader_id"])
    & validation_data["location_match"].eq(1)
    & validation_data["sequence_match"].eq(1)
    & validation_data["delayed_indicator"].eq(0)
    & ~duplicate_check
)

# Review both security decisions
print("Traditional rule decisions:")
print(validation_data["traditional_flagged"].value_counts())

print("\nZero Trust decisions:")
print(validation_data["zta_flagged"].value_counts())

# Confirm that the validation groups do not overlap
overlap = set(rfid_validation["bag_id"]) & set(rfid_remaining["bag_id"])

print("\nOverlapping baggage journeys:", len(overlap))


Traditional rule decisions:
traditional_flagged
False    8549
True      215
Name: count, dtype: int64

Zero Trust decisions:
zta_flagged
False    8455
True      309
Name: count, dtype: int64

Overlapping baggage journeys: 0


Step 5: Compare the RFID Results

The two methods are compared with the known scan labels. The notebook counts correct detections, false alarms, and missed abnormal scans. It also calculates accuracy, precision, recall, and F1 score. This is another test using the simulated RFID records.


In [5]:

# Step 5: Compare Traditional Rules and Zero Trust performance

# Load the evaluation tools
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Set the actual abnormal scan labels
actual_labels = validation_data["final_label"].eq("Abnormal")

# Set the security predictions
traditional_predictions = validation_data["traditional_flagged"]
zta_predictions = validation_data["zta_flagged"]

# Create the results table
results = []

# Evaluate both security approaches
for name, predictions in [
    ("Traditional Rules", traditional_predictions),
    ("Zero Trust", zta_predictions)
]:

    # Calculate the confusion matrix
    tn, fp, fn, tp = confusion_matrix(
        actual_labels,
        predictions,
        labels=[False, True]
    ).ravel()

    # Calculate the detection measures
    results.append({
        "Approach": name,
        "Accuracy": accuracy_score(actual_labels, predictions),
        "Precision": precision_score(actual_labels, predictions),
        "Recall": recall_score(actual_labels, predictions),
        "F1 Score": f1_score(actual_labels, predictions),
        "False Positives": fp,
        "False Negatives": fn,
        "True Positives": tp,
        "True Negatives": tn
    })

# Create the H1 comparison table
h1_validation_results = pd.DataFrame(results)

# Display the results
print("H1 Validation Performance Comparison:")
print(h1_validation_results.round(4).to_string(index=False))

# Review detection by RFID scan condition
print("\nDetected scans by condition:")

print(
    validation_data.groupby("scan_status")[
        ["traditional_flagged", "zta_flagged"]
    ].sum().astype(int)
)


H1 Validation Performance Comparison:
         Approach  Accuracy  Precision  Recall  F1 Score  False Positives  False Negatives  True Positives  True Negatives
Traditional Rules    0.9869        1.0  0.6515    0.7890                0              115             215            8434
       Zero Trust    0.9976        1.0  0.9364    0.9671                0               21             309            8434

Detected scans by condition:
                   traditional_flagged  zta_flagged
scan_status                                        
Delayed                              0           46
Denial of Service                    0            0
Duplicate                            0           48
Invalid Location                    65           65
Normal                               0            0
Out of Sequence                     54           54
Spoofed Reader                      96           96


Step 6: Check the IoT23 Network Records

The network records are checked for unusually frequent connections from the same source within one minute. The connection counts are compared with the chosen alert limit and the known traffic labels. These results are reported separately because network traffic is not baggage scan data.


In [6]:

# Step 6: Evaluate IoT23 network security activity

# Select the network fields
network_activity = iot23_data[
    ["ts", "id.orig_h", "label"]
].copy()

# Convert timestamps to numbers
network_activity["ts"] = pd.to_numeric(
    network_activity["ts"],
    errors="coerce"
)

# Remove records without usable timestamps or sources
network_activity = network_activity.dropna(
    subset=["ts", "id.orig_h"]
)

# Group connections into one minute intervals
network_activity["minute"] = (
    network_activity["ts"] // 60
).astype("int64")

# Count connections for each source per minute
network_activity["connection_count"] = (
    network_activity.groupby(["id.orig_h", "minute"])["ts"]
    .transform("size")
)

# Set a preliminary network alert threshold
alert_threshold = 30

# Flag sources with high connection activity
network_activity["network_alert"] = (
    network_activity["connection_count"] >= alert_threshold
)

# Review network alert decisions
print("Network records evaluated:", len(network_activity))

print("\nNetwork alert decisions:")
print(network_activity["network_alert"].value_counts())

# Compare alerts with recorded network labels
print("\nNetwork alerts by traffic label:")
print(pd.crosstab(
    network_activity["label"],
    network_activity["network_alert"]
))

# Calculate network detection performance
network_actual = network_activity["label"].eq("Malicious")
network_prediction = network_activity["network_alert"]

tn, fp, fn, tp = confusion_matrix(
    network_actual,
    network_prediction,
    labels=[False, True]
).ravel()

print("\nNetwork detection performance:")
print("Accuracy:", round(accuracy_score(network_actual, network_prediction), 4))
print("Precision:", round(precision_score(network_actual, network_prediction, zero_division=0), 4))
print("Recall:", round(recall_score(network_actual, network_prediction), 4))
print("F1 Score:", round(f1_score(network_actual, network_prediction), 4))
print("False Positives:", fp)
print("False Negatives:", fn)


Network records evaluated: 23145

Network alert decisions:
network_alert
True     14574
False     8571
Name: count, dtype: int64

Network alerts by traffic label:
network_alert  False  True 
label                      
Benign          1880     43
Malicious       6691  14531

Network detection performance:
Accuracy: 0.7091
Precision: 0.997
Recall: 0.6847
F1 Score: 0.8119
False Positives: 43
False Negatives: 6691


In [7]:

### Save the H1 Validation and IoT23 Results

The RFID validation results and IoT23 network monitoring results are saved separately. This preserves the distinction between baggage scan anomaly detection and network threat identification.

The original H1 results remain unchanged.


SyntaxError: invalid syntax (3089904467.py, line 3)

In [ ]:

# Save the H1 validation and IoT23 results

# Set the H1 results folder
results_folder = main_folder / "07_H1_Results"

# Create the folder if needed
results_folder.mkdir(parents=True, exist_ok=True)

# Save the RFID performance comparison
h1_validation_results.to_csv(
    results_folder / "H1_Additional_RFID_Validation.csv",
    index=False
)

# Save the network alert decisions
network_activity.to_csv(
    results_folder / "H1_IoT23_Network_Alert_Results.csv",
    index=False
)

# Save the IoT23 performance measures
network_metrics = pd.DataFrame([{
    "Accuracy": accuracy_score(network_actual, network_prediction),
    "Precision": precision_score(network_actual, network_prediction),
    "Recall": recall_score(network_actual, network_prediction),
    "F1 Score": f1_score(network_actual, network_prediction),
    "False Positives": fp,
    "False Negatives": fn,
    "True Positives": tp,
    "True Negatives": tn
}])

network_metrics.to_csv(
    results_folder / "H1_IoT23_Network_Metrics.csv",
    index=False
)

print("H1 supplemental results saved successfully.")
print("Results folder:", results_folder)


H1 supplemental results saved successfully.
Results folder: /workspaces/Aviation-Baggage-Cybersecurity-Research/07_H1_Results
